<img src="../img/hero-machine.jpg" width="100%" style="max-height:240px;object-fit:cover;border-radius:8px;">

# Predicción de Parada de Máquina
## Sistema de mantenimiento predictivo industrial por capas

**Memoria del Proyecto Final · Bootcamp de Data Science**

Autora: **Nora Galparsoro** · Junio 2026

---

> Sistema de mantenimiento predictivo que integra **cuatro fuentes de datos industriales**
> en una **arquitectura de escalada por capas** (eléctrica → mecánica → proceso), replicando
> cómo se monitoriza una máquina-herramienta real: detección continua barata, confirmación
> sin hardware extra y, finalmente, clasificación del tipo de fallo con orden de mantenimiento.

## Resumen ejecutivo

El objetivo es **anticipar fallos en máquinas-herramienta** para reducir las paradas no
planificadas. En lugar de un único modelo, se diseñó un **pipeline de 3 capas** donde cada
capa usa una fuente de datos distinta y solo se activa si la anterior alerta:

| Capa | Fuente / dataset | Modelo | Rol |
|---|---|---|---|
| **1 — Eléctrica** | SPARK (11 máquinas, 1 min) | Isolation Forest | Detección continua, 100 % del parque |
| **2 — Mecánica** | KIT (prototipo) → CNC (producción) | Random Forest | Confirma fallo mecánico, sin hardware extra |
| **3 — Proceso** | AI4I 2020 (10.000 muestras) | Stacking LGBM+RF | Clasifica el **tipo** de fallo → acción |

**Modelo estrella (Capa 3):** `StackingClassifier` (LightGBM + Random Forest → Logistic
Regression) con umbral optimizado.

| Recall | Precision | F1 | ROC-AUC | Umbral |
|---|---|---|---|---|
| **0.853** | 0.967 | 0.906 | 0.978 | 0.9858 |

**Impacto económico (Capa 3):** para una pyme de 5 máquinas con un gasto de mantenimiento
de referencia de ~1 M€/año, el modelo evita ~81 % del coste por fallos → **≈ 0,83 M€/año**
de ahorro.

**Aportación metodológica clave:** se documenta con rigor **qué es y qué no es comparable**.
Solo la Capa 3 tiene métricas de test (n=2000) que sostienen una cifra económica; las capas 1
y 2 se evalúan por su rol, cobertura y coste, **no** convirtiendo en € unos recalls medidos
sobre muestras minúsculas (n=18–33) que producirían conclusiones falsas.

## Índice

1. Introducción y contexto
2. Objetivos e hipótesis
3. Los datos: cuatro datasets industriales
4. Metodología: arquitectura por capas
5. Análisis exploratorio (EDA)
6. Preprocesado y *feature engineering*
7. Modelado iterativo
8. Resultados de la Capa 3 (modelo de producción)
9. Capas 1 y 2: evaluación honesta
10. Sistema integrado: `EscalationPipeline`
11. Análisis económico
12. Limitaciones y honestidad metodológica
13. Conclusiones
14. Trabajo futuro
15. Reproducibilidad y stack técnico

## 1. Introducción y contexto

En la industria del mecanizado, una **parada no planificada** de una máquina-herramienta
(CNC, fresadora, centro de mecanizado) detiene la producción y obliga a una reparación de
urgencia. El coste de cada evento varía mucho según la criticidad del equipo: la literatura
de mantenimiento sitúa el *downtime* en fabricación entre **5.000 y 50.000 €/evento**.

El **mantenimiento predictivo** (PdM) busca anticipar esos fallos a partir de los datos que
la máquina ya genera (corriente, vibración, temperatura, parámetros de proceso), para
intervenir *antes* de la avería y de forma planificada.

El reto realista no es solo "entrenar un buen clasificador", sino que **una misma planta tiene
máquinas con distinta instrumentación**: unas solo tienen una pinza amperimétrica externa,
otras exponen las señales internas del controlador por OPC-UA, y solo algunas están conectadas
a un MES/SCADA con los parámetros de proceso. Un sistema útil debe **degradar con elegancia**:
aportar valor con la instrumentación mínima y ganar precisión a medida que hay más sensores.

Ese es el principio que vertebra todo el proyecto.

## 2. Objetivos e hipótesis

**Objetivo general.** Construir un sistema de mantenimiento predictivo que detecte fallos
inminentes en máquinas-herramienta y, cuando sea posible, **clasifique el tipo de fallo** para
emitir una acción de mantenimiento concreta.

**Objetivos específicos.**
- Maximizar el **recall** de la clase fallo (no escapar averías) manteniendo una precisión alta
  (pocas alarmas falsas), sobre un dataset **fuertemente desbalanceado** (~3,4 % de fallos).
- Integrar fuentes de datos heterogéneas (eléctrica, mecánica, proceso) en una **arquitectura
  por capas** desplegable en planta sin hardware adicional.
- Cuantificar el **impacto económico** de forma defendible y honesta.

**Hipótesis de partida.**
1. Los parámetros de proceso (torque, desgaste, temperatura, velocidad) contienen señal
   suficiente para predecir el fallo binario con alto recall. → *Capa 3 (AI4I).*
2. Las señales internas del controlador CNC permiten confirmar un fallo mecánico **sin instalar
   sensores externos**. → *Capa 2 (CNC).*
3. El consumo eléctrico agregado sirve como **detector temprano no supervisado** aplicable a
   cualquier máquina. → *Capa 1 (SPARK).*

## 3. Los datos: cuatro datasets industriales

Cada capa se apoya en un dataset distinto, elegido por representar una fuente de señal real:

| Dataset | Fuente | Tamaño | Etiquetas | Rol |
|---|---|---|---|---|
| **AI4I 2020** | UCI ML Repository | 10.000 muestras | Fallo binario + 5 tipos (TWF/HDF/PWF/OSF/RNF) | Capa 3 |
| **CNC Mill** | Kaggle (Tool Wear Detection) | 18 experimentos | Desgaste / acabado | Capa 2 (producción) |
| **KIT Industrial** | FIZ Karlsruhe (DOI 10.35097/hvvwn1kfwf7qt48z) | 33 experimentos, 500 Hz | Anomalía real (5 tipos) | Capa 2 (prototipo) |
| **SPARK TEC** | FIZ Karlsruhe (DOI 10.35097/bjdg3m3rg5jv3skk) | 11 máquinas, 1 min, 1 año | *Sin etiquetas* | Capa 1 |

**Tipos de fallo (AI4I).** El dataset distingue cinco modos de fallo, cada uno con una firma
distinta en los sensores:

<img src="../img/tipos_fallo_oscuro_2.png" width="80%">

- **TWF** — *Tool Wear Failure*: desgaste de herramienta.
- **HDF** — *Heat Dissipation Failure*: fallo de disipación de calor.
- **PWF** — *Power Failure*: potencia fuera de rango.
- **OSF** — *Overstrain Failure*: sobreesfuerzo.
- **RNF** — *Random Failure*: fallo aleatorio (sin firma clara).

> **Procedencia y reproducción** detallada (cómo descargar y regenerar cada dataset) en el
> `README.md` del proyecto.

## 4. Metodología: arquitectura por capas

El sistema implementa una **escalada**: cada capa solo se activa si la anterior ha alertado,
de modo que se usa la fuente de datos más barata primero y se reserva la más costosa (MES)
para confirmar y diagnosticar.

<img src="../img/Diagnostico_integrado.png" width="92%">

```
CAPA 1 — Detección eléctrica          [siempre activa, 1 min]
    Isolation Forest sobre señales SPARK → anomaly_score   · cobertura 100 %
         ↓ si score < umbral
CAPA 2 — Confirmación mecánica         [se activa si Capa 1 alerta]
    Random Forest sobre señales internas CNC (OPC-UA)      · sin hardware extra
         ↓ si P(fallo mecánico) > umbral
CAPA 3 — Clasificación de fallo        [se activa si Capa 2 confirma]
    StackingClassifier sobre parámetros de proceso AI4I    · tipo de fallo + acción
```

**Por qué por capas y no un único modelo:**
- **Útil desde la Capa 1.** Aporta valor aunque solo exista la capa eléctrica.
- **Degradación elegante.** Si falla un sensor, las demás capas siguen operando.
- **Coste-beneficio progresivo.** Cada capa añade confianza y detalle justificando su coste.

El proyecto se desarrolló en **7 fases** (carpetas `01`→`07`), cada una con un notebook por
dataset: EDA → preprocesado → modelos iniciales → evaluación → modelos afinados → modelos
finales por capa → integración.

## 5. Análisis exploratorio (EDA)

**AI4I (Capa 3).** El hallazgo dominante es el **desbalanceo extremo**: solo el ~3,4 % de las
muestras son fallos. Esto condiciona toda la estrategia posterior (métricas, umbral, pesos de
clase). Las variables más discriminantes resultaron ser el **torque**, el **desgaste de
herramienta** y la **diferencia de temperatura** proceso–aire.

**SPARK (Capa 1).** Un año de consumo eléctrico a 1 minuto de 11 máquinas. Sin etiquetas de
fallo, el EDA se centró en la **consistencia entre máquinas** y la detección de anomalías no
supervisada. La tasa de anomalías queda estable en torno al *contamination* fijado.

<table><tr>
<td><img src="../img/spark_comparativa.png" width="100%"></td>
<td><img src="../img/spark_pca.png" width="100%"></td>
</tr><tr>
<td align="center"><sub>Comparativa de consumo / anomalías entre máquinas SPARK</sub></td>
<td align="center"><sub>PCA de una máquina con scores de Isolation Forest</sub></td>
</tr></table>

## 6. Preprocesado y *feature engineering*

**AI4I (Capa 3) — 10 features.** A las variables originales se añadieron features de ingeniería
derivadas del dominio físico:

| Feature | Definición | Por qué |
|---|---|---|
| `Power` | `Torque · Rot_speed · 2π/60` | Potencia mecánica real del husillo |
| `Temp_diff` | `Process_temp − Air_temp` | Salto térmico (clave para HDF) |
| `Wear_torque` | `Tool_wear · Torque` | Interacción desgaste×esfuerzo (OSF/TWF) |
| `wear_ratio` | `Tool_wear / TWF_MIN[Type]` | Normaliza el desgaste por el umbral de cada tipo de máquina (L=200, M=220, H=240) |

La feature **`wear_ratio`** se introdujo específicamente para atacar los falsos negativos de
tipo TWF (ver §8.1). El preprocesado se completa con `LabelEncoder` para el tipo de máquina y
`StandardScaler`, con un *split* estratificado 80/20.

**SPARK (Capa 1).** Agregación de la señal de 5 s → 1 min, detección de *gaps* y escalado por
máquina antes del Isolation Forest.

**KIT/CNC (Capa 2).** Cada experimento se resume en *features* estadísticas (media, std, máx,
mín) por canal del controlador → un vector por experimento.

## 7. Modelado iterativo

El modelado de la Capa 3 (AI4I) siguió una progresión deliberada, documentada en los notebooks
`03_*` → `05_*` → `06_*`:

1. **Modelos base** (`03_01`): Logistic Regression, Random Forest, XGBoost, LightGBM, SVM —
   comparativa con métricas de partida.
2. **Afinado con Optuna** (`05_01_1`–`05_01_3`): búsqueda de hiperparámetros para LightGBM, RF
   y SVM, con `scale_pos_weight` / `class_weight` para el desbalanceo.
3. **Ensamblado** (`05_01_4`): `StackingClassifier` con LightGBM + RF como *base learners* y
   Logistic Regression como meta-modelo → **modelo de producción**.
4. **Exploraciones complementarias:**
   - *Dos etapas* (`05_01_6`): detección binaria + clasificación de tipo de fallo.
   - *No supervisado* (`05_01_5`): PCA, t-SNE, clustering, Isolation Forest.
   - *Tool tracking* (`05_01_7`): modelo reentrenable de Capa 2 con señales CNC internas.

**Decisión de diseño — el umbral.** Con datos tan desbalanceados, el umbral 0.5 por defecto es
inadecuado. Se optimizó para **garantizar Recall ≥ 0.85 maximizando F1**, lo que llevó a un
umbral alto (**0.9858**): el modelo solo declara fallo cuando está muy seguro, minimizando
falsas alarmas sin perder recall.

## 8. Resultados de la Capa 3 (modelo de producción)

La Capa 3 es la única evaluada con todas las garantías: **conjunto de test independiente de
2.000 muestras**. Cargamos sus métricas oficiales desde `models/model_config.yaml`.

In [ ]:
import os, yaml
import pandas as pd

MODELS = '../models'
with open(os.path.join(MODELS, 'model_config.yaml')) as f:
    cfg = yaml.safe_load(f)

p = cfg['performance']
print('Capa 3 — StackingClassifier (LightGBM + RF), AI4I 2020 · test n=2000')
print(f"  Umbral de producción : {cfg['threshold']}")
print(f"  Recall               : {p['recall']:.3f}")
print(f"  Precision            : {p['precision']:.3f}")
print(f"  F1                   : {p['f1']:.3f}")
print(f"  ROC-AUC              : {p['roc_auc']:.3f}")
print(f"  Falsos negativos     : {p['fn']} / 68 fallos")
print(f"  Falsos positivos     : {p['fp']} / 2000 muestras")

### 8.1 Análisis de falsos negativos

Con el umbral de producción, el modelo **escapa 10 fallos de 68** en test. El análisis de sus
probabilidades reveló dos grupos:

- **Grupo A — *catchables* (3 fallos, prob ≥ 0.25):** detectables bajando el umbral.
- **Grupo B — *invisibles* (7 fallos, prob < 0.25):** todos son **TWF** (*Tool Wear Failure*)
  con desgaste muy alto (~202 min de media) pero **torque moderado**.

| | FN invisibles | Operación normal |
|---|---|---|
| Tool Wear medio | **202 min** | 107 min |
| Torque medio | 35.8 Nm | 39.6 Nm |

**Causa raíz.** El TWF se dispara con un umbral de desgaste **aleatorio e individual por
herramienta**, no observable en los sensores. La feature `wear_ratio` se diseñó para mitigarlo,
pero al ser casi linealmente dependiente de `Tool_wear` + `Type` (que el modelo ya tenía) la
mejora fue marginal (FN 11→10). **Conclusión honesta:** esos 7 TWF (~2 % de los fallos) son
inherentemente difíciles con las features disponibles; detectarlos requeriría un identificador
de herramienta o features temporales. No es un defecto de calibración del modelo.

### 8.2 Interpretabilidad — SHAP

Se aplicó `TreeExplainer` (SHAP) sobre el LightGBM. Las features de mayor impacto coinciden con
la intuición física: **torque**, **desgaste de herramienta** y las interacciones derivadas
(`Wear_torque`, `Temp_diff`), lo que da confianza en que el modelo aprende mecanismos reales de
fallo y no artefactos.

## 9. Capas 1 y 2: evaluación honesta

Las capas 1 y 2 **no se pueden evaluar como la Capa 3** porque su *ground truth* es distinto:

- **Capa 1 (SPARK):** no hay etiquetas de fallo. Solo es evaluable la **consistencia** del
  detector (la tasa de anomalías debe ser estable entre máquinas y rondar el *contamination*).
- **Capa 2 (KIT/CNC):** hay etiquetas, pero con **n=33 / n=18 experimentos** la validación es
  *Leave-One-Out* y sus métricas tienen un **intervalo de confianza enorme**.

Cargamos las métricas LOO reales de los modelos de Capa 2:

In [ ]:
import pickle

with open(os.path.join(MODELS, 'kit_model.pkl'), 'rb') as f:
    kit = pickle.load(f)
with open(os.path.join(MODELS, 'cnc_model.pkl'), 'rb') as f:
    cnc = pickle.load(f)

tabla = pd.DataFrame({
    'KIT (prototipo, n=33)':  kit['metrics_loo'],
    'CNC (cold-start, n=18)': cnc['metrics_loo'],
}).T
print(tabla.to_string())
print()
print('⚠ El recall=1.0 del CNC NO es overfitting clásico (es LOO, cada experimento se predice')
print('  fuera de entrenamiento), pero con ~5 fallos reales sobre n=18 es estadísticamente vacío:')
print('  acertar 5 de 5 es plausible por azar. Su valor es de prototipo, no de cifra fiable.')

**Hoja de ruta de la Capa 2.** El modelo arranca con el *baseline* KIT (sensores externos,
prototipo), se sustituye en planta por el modelo **CNC** (señales internas del controlador, sin
hardware extra) usando los 18 experimentos como *cold-start*, y se **reentrena de forma
incremental** con datos reales hasta alcanzar una muestra suficiente para métricas fiables.

<img src="../img/roadmap.png" width="78%">

## 10. Sistema integrado: `EscalationPipeline`

El notebook `07_Sistema_Industrial` implementa la clase central del sistema. Su método
`monitor(signals)` recibe las señales disponibles y aplica la lógica de escalada, devolviendo
un `AlarmResult` con el nivel de alarma, la causa y la **acción de mantenimiento**:

```
anomaly_score < umbral_L1?
    NO  → Normal, no escalar
    SÍ  → ¿Datos mecánicos?  NO → Alarma Capa 1 (anomalía eléctrica)
                             SÍ → P(mecánico) > umbral_L2?
                                     NO → Problema eléctrico aislado
                                     SÍ → ¿Datos de proceso? NO → Alarma Capa 2 (mecánico)
                                                             SÍ → Clasificar tipo + Capa 3
```

Se validó sobre **4 escenarios** representativos (operación normal, anomalía eléctrica de red,
fallo en cascada por desgaste crítico, y máquina *legacy* solo con Capa 1), con un **dashboard
de operario** estilo SCADA que muestra el estado de cada capa y la acción recomendada.

## 11. Análisis económico

Se modela el coste anual en función de los dos errores, calibrado para una **pyme de criticidad
media**:

- **Falso negativo (FN)** — fallo no detectado → parada imprevista: **~3.000 €/fallo**.
- **Falso positivo (FP)** — alarma falsa → revisión innecesaria: **~300 €/alarma**.

Se proyecta a una **pyme de 5 máquinas**, cuyo gasto de mantenimiento de referencia (todos los
fallos sin sistema) es de ~1 M€/año. **Solo se cuantifica la Capa 3**, la única con recall de
test (n=2000); ver §12.

In [ ]:
import matplotlib.pyplot as plt

COST_FN, COST_FP = 3_000, 300
N_FAILURES_YEAR, N_PREDICTIONS, N_MACHINES = 68, 1000, 5   # pyme: 5 máquinas → baseline ~1M€

recall, precision = p['recall'], p['precision']
baseline = N_FAILURES_YEAR * COST_FN * N_MACHINES
fn = int(N_FAILURES_YEAR * (1 - recall)) * N_MACHINES
fp = int(N_PREDICTIONS * (1 - precision) * recall) * N_MACHINES
coste = fn * COST_FN + fp * COST_FP
ahorro = baseline - coste

print(f'Coste sin sistema (baseline) : €{baseline:,.0f}/año')
print(f'Coste con Capa 3             : €{coste:,.0f}/año  (FN={fn}, FP={fp})')
print(f'AHORRO Capa 3                : €{ahorro:,.0f}/año  ({ahorro/baseline:.0%} reducción)')

fig, ax = plt.subplots(figsize=(7, 2.6))
ax.barh(['Sin sistema', 'Con Capa 3'], [baseline, coste], color=['tomato', '#2ecc71'],
        edgecolor='white')
for i, v in enumerate([baseline, coste]):
    ax.text(v, i, f' €{v/1e3:.0f}K', va='center', fontsize=9)
ax.set_xlabel('Coste anual estimado (€) — pyme de 5 máquinas')
ax.set_title('Capa 3 — coste con sistema vs sin sistema')
plt.tight_layout(); plt.show()

## 12. Limitaciones y honestidad metodológica

Esta sección es deliberadamente explícita, porque es donde el proyecto aporta criterio:

1. **No existe evaluación *end-to-end* real del sistema.** Ningún dataset contiene las señales
   de las tres capas a la vez sobre las mismas máquinas. Las capas detectan fallos de **dominios
   distintos**, así que **no hay un "recall combinado" honesto**.

2. **Los recalls de las capas NO son comparables en €.** Meter las tres capas en una tabla de
   ahorro produce lecturas falsas. Caso real detectado en este proyecto: *"Capa 1+2 ahorra más
   que el sistema completo"*. Es un **artefacto**: el recall=1.0 del CNC viene de un LOO con
   n=18 (FN=0), y como el coste lo domina el FN, esa configuración "gana" sin ser mejor. Por eso
   **solo la Capa 3 (test n=2000) recibe una cifra en €**; las capas 1 y 2 se justifican por
   **cobertura y coste de instalación**, no por ahorro.

3. **La Capa 2 está infra-entrenada** (n=18–33). Sus métricas son orientativas; el diseño
   correcto es el reentrenamiento incremental en planta.

4. **Sensibilidad al modelo de costes.** El resultado depende de la relación FN/FP. Con FN ≫ FP
   (caso de equipos críticos) "manda el recall"; si las falsas alarmas fueran más caras que los
   fallos, el balance se invertiría. Las cifras se presentan **con sus supuestos explícitos**.

5. **Límite intrínseco en TWF** (§8.1): ~2 % de fallos no son detectables con las features
   actuales por depender de un umbral aleatorio por herramienta.

## 13. Conclusiones

- Se construyó un **sistema de mantenimiento predictivo por capas** que integra cuatro fuentes
  de datos industriales y replica la operativa real de una planta, con **degradación elegante**.
- El **modelo de producción (Capa 3)** alcanza **Recall 0.853 / F1 0.906 / AUC 0.978** sobre
  test independiente, cumpliendo el objetivo de alto recall con mínimas falsas alarmas.
- Para una **pyme de 5 máquinas** (~1 M€ de mantenimiento), la Capa 3 ahorra **≈ 0,83 M€/año
  (~81 %)** — la única cifra económica defendible del sistema.
- Se validan las tres hipótesis de partida, con la **matización honesta** de que las capas 1 y 2
  quedan a nivel de prototipo por falta de datos etiquetados suficientes.
- El proyecto demuestra no solo capacidad de modelado, sino **criterio para no sobre-vender**:
  distingue qué métricas son fiables y cuáles no, y por qué.

## 14. Trabajo futuro

- **Reentrenamiento incremental de la Capa 2** en planta hasta alcanzar n suficiente (decenas/
  cientos de fallos) para métricas con intervalo de confianza estrecho.
- **Etiquetado de la Capa 1** mediante correlación con paradas reales, para poder medir su
  recall y, entonces sí, cuantificar su aporte económico.
- **Features temporales / identificador de herramienta** para atacar los TWF invisibles.
- **Despliegue real** del `EscalationPipeline` contra un CMMS (creación automática de órdenes de
  trabajo) y un controlador vía OPC-UA.

## 15. Reproducibilidad y stack técnico

```bash
pip install -r requirements.txt          # Python 3.12

# App de demo interactiva
python -m streamlit run src/app.py

# Regenerar artefactos
python src/data_processing.py            # data/processed/
python src/training.py                   # models/final_model.pkl + model_config.yaml
python src/evaluation.py                 # informe de evaluación
```

**Stack:** `scikit-learn` · `LightGBM` · `XGBoost` · `Optuna` · `SHAP` · `pandas` · `numpy` ·
`scipy` · `matplotlib` · `seaborn` · `Streamlit` · `pyarrow`.

**Estructura:** notebooks por fase (`01`→`07`) y dataset; código reutilizable en `src/`
(`paths`, `utils`, `kit`, `spark`, `data_processing`, `training`, `evaluation`); artefactos en
`models/`; datos en `data/`.

---

### Referencias

- AI4I 2020 Predictive Maintenance Dataset — *UCI Machine Learning Repository*.
- CNC Mill Tool Wear — *Kaggle (System-level Manufacturing and Automation Research Testbed)*.
- KIT Industrial / SPARK TEC — *FIZ Karlsruhe* (DOI 10.35097/hvvwn1kfwf7qt48z y
  10.35097/bjdg3m3rg5jv3skk).